# 02 - Synthetic Corpus Generation

This notebook generates synthetic parallel data using Gemma as a teacher model through back-translation and domain-specific dialogue synthesis.

**Platform:** Kaggle (recommended) or Google Colab
**GPU Required:** Yes (T4 minimum, A100 recommended for faster generation)
**Estimated Time:** 2-4 hours for ~10,000 pairs per language direction

## Techniques Used:
1. **Back-translation** - Translate source→target, then verify with target→source
2. **Domain dialogue synthesis** - Generate humanitarian/medical/legal domain conversations
3. **Paraphrase augmentation** - Create variations of seed sentences

## Prerequisites:
- Run `01_download_seed_data.ipynb` first OR run the download cell below
- Hugging Face account with Gemma access (accept license at https://huggingface.co/google/gemma-2-27b-it)

## 1. Environment Setup

Run this cell first to install dependencies and configure the environment.

In [ ]:
# ============================================================
# ENVIRONMENT SETUP
# ============================================================
# Detect runtime environment
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

print(f"Running in: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'}")

# Install dependencies
!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q datasets tqdm

# For Colab: Mount Google Drive for persistent storage
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/Daraja'
elif IN_KAGGLE:
    BASE_DIR = '/kaggle/working'
else:
    BASE_DIR = '..'

print(f"Base directory: {BASE_DIR}")

# Hugging Face login for gated models (Gemma requires acceptance)
from huggingface_hub import login

if IN_KAGGLE:
    # Use Kaggle secrets
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in via Kaggle secret!")
elif IN_COLAB:
    # Use Colab secrets
    from google.colab import userdata
    try:
        hf_token = userdata.get('HF_TOKEN')
        login(token=hf_token)
        print("Logged in via Colab secret!")
    except:
        import getpass
        token = getpass.getpass("HF Token: ")
        login(token=token)
        print("Logged in successfully!")
else:
    # Local - try to use cached login or prompt
    try:
        from huggingface_hub import whoami
        user = whoami()
        print(f"Logged in as: {user['name']}")
    except:
        import getpass
        print("Please enter your Hugging Face token:")
        token = getpass.getpass("HF Token: ")
        login(token=token)
        print("Logged in successfully!")

In [ ]:
# ============================================================
# IMPORTS AND CONFIGURATION
# ============================================================
import json
import torch
from pathlib import Path
from tqdm.auto import tqdm
from datetime import datetime

# Configuration - EDIT THESE FOR YOUR RUN
SOURCE_LANG = "so"           # Source language code
TARGET_LANG = "sw"           # Target language code
TEACHER_MODEL = "google/gemma-2-2b-it"  # Smaller model that fits on T4 GPU
MAX_PAIRS = 2000             # Reduced for faster generation (increase if you have time)
BATCH_SIZE = 1               # Batch size for generation

# Derived paths
SEED_DIR = Path(BASE_DIR) / 'pipeline' / 'data' / 'seed'
OUTPUT_DIR = Path(BASE_DIR) / 'pipeline' / 'data' / 'synthetic' / f'{SOURCE_LANG}-{TARGET_LANG}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Configuration:")
print(f"  Source: {SOURCE_LANG}")
print(f"  Target: {TARGET_LANG}")
print(f"  Teacher model: {TEACHER_MODEL}")
print(f"  Max pairs: {MAX_PAIRS:,}")
print(f"  Output: {OUTPUT_DIR}")
print(f"\nGPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Load Teacher Model

Loading Gemma 27B with 4-bit quantization to fit in GPU memory.

In [ ]:
# ============================================================
# LOAD TEACHER MODEL (OPTIONAL - only needed for domain dialogue generation)
# ============================================================
# Since we have real parallel data from NLLB, we can skip model loading
# unless you want to generate domain-specific dialogues.

LOAD_TEACHER_MODEL = False  # Set to True if you want to generate domain dialogues

if LOAD_TEACHER_MODEL:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    print(f"Loading teacher model: {TEACHER_MODEL}")
    print(f"This may take a few minutes...")

    # 4-bit quantization config for memory efficiency
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL)

    # Load model with quantization
    model = AutoModelForCausalLM.from_pretrained(
        TEACHER_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )

    print(f"Model loaded successfully!")
    print(f"Model memory: {model.get_memory_footprint() / 1e9:.2f} GB")
else:
    print("Skipping teacher model loading (LOAD_TEACHER_MODEL = False)")
    print("Using NLLB parallel data directly - no model needed!")
    print("\nThis saves GPU memory and time.")
    model = None
    tokenizer = None

In [ ]:
# ============================================================
# TRANSLATION HELPER FUNCTIONS (only used if generating domain dialogues)
# ============================================================
# Language name mappings
LANG_NAMES = {
    "so": "Somali",
    "sw": "Swahili", 
    "ti": "Tigrinya",
    "ar": "Arabic",
    "prs": "Dari",
    "tr": "Turkish",
    "en": "English"
}

SOURCE_NAME = LANG_NAMES.get(SOURCE_LANG, SOURCE_LANG)
TARGET_NAME = LANG_NAMES.get(TARGET_LANG, TARGET_LANG)

def generate_text(prompt, max_new_tokens=256, temperature=0.7, top_p=0.9):
    """Generate text using the teacher model."""
    if model is None:
        raise RuntimeError("Model not loaded. Set LOAD_TEACHER_MODEL = True")
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response[len(tokenizer.decode(inputs['input_ids'][0], skip_special_tokens=True)):].strip()
    return response

def translate(text, source_lang, target_lang, domain=None):
    """Translate text from source to target language."""
    source_name = LANG_NAMES.get(source_lang, source_lang)
    target_name = LANG_NAMES.get(target_lang, target_lang)
    domain_context = f" This is {domain} domain text." if domain else ""
    
    prompt = f"""<start_of_turn>user
Translate the following {source_name} text to {target_name}.{domain_context}
Only output the translation, nothing else.

{text}<end_of_turn>
<start_of_turn>model
"""
    return generate_text(prompt)

def generate_domain_sentence(lang, domain, context):
    """Generate a domain-specific sentence."""
    lang_name = LANG_NAMES.get(lang, lang)
    
    prompt = f"""<start_of_turn>user
Generate a short {lang_name} sentence that a person might say during: {context}
Only output the sentence in {lang_name}, nothing else.<end_of_turn>
<start_of_turn>model
"""
    return generate_text(prompt, max_new_tokens=100)

print(f"Language pair: {SOURCE_NAME} -> {TARGET_NAME}")

## 3. Load Seed Data

Loading seed sentences for back-translation. If seed data doesn't exist, we'll download it.

In [ ]:
# ============================================================
# LOAD SEED DATA - NLLB Parallel Corpus
# ============================================================
from pathlib import Path

# Kaggle dataset path (your uploaded dataset)
source_file = Path("/kaggle/input/datasets/jeremiahsakuda/daraja-seed-data/NLLB.so-sw.so")
target_file = Path("/kaggle/input/datasets/jeremiahsakuda/daraja-seed-data/NLLB.so-sw.sw")

# Fallback to local paths if not on Kaggle
if not source_file.exists():
    source_file = Path(BASE_DIR) / "so-sw-data" / "NLLB.so-sw.so"
    target_file = Path(BASE_DIR) / "so-sw-data" / "NLLB.so-sw.sw"

if not source_file.exists():
    source_file = Path(BASE_DIR) / "pipeline" / "data" / "seed" / "so-sw" / "seed.so"
    target_file = Path(BASE_DIR) / "pipeline" / "data" / "seed" / "so-sw" / "seed.sw"

if not source_file.exists():
    raise FileNotFoundError("Seed data not found! Upload dataset to Kaggle.")

print(f"Loading from: {source_file}")

with open(source_file, 'r', encoding='utf-8') as f:
    source_sentences = [line.strip() for line in f if line.strip()]

with open(target_file, 'r', encoding='utf-8') as f:
    target_sentences_reference = [line.strip() for line in f if line.strip()]

print(f"Loaded {len(source_sentences):,} Somali sentences")
print(f"Loaded {len(target_sentences_reference):,} Swahili sentences")

# Limit to MAX_PAIRS
source_sentences = source_sentences[:MAX_PAIRS]
target_sentences_reference = target_sentences_reference[:MAX_PAIRS]

print(f"\nUsing {len(source_sentences):,} pairs for training")
print(f"\nSample parallel pairs:")
for i in range(min(3, len(source_sentences))):
    print(f"\n  Somali:  {source_sentences[i][:70]}...")
    print(f"  Swahili: {target_sentences_reference[i][:70]}...")

## 4. Generate Back-Translation Pairs

This is the main generation loop. We translate source→target and optionally verify with back-translation.

In [ ]:
# ============================================================
# PREPARE TRAINING PAIRS FROM NLLB DATA
# ============================================================
# Since we already have parallel data, we can use it directly!
# No need to generate translations with Gemma - we have real ones.

import time

print(f"Preparing {len(source_sentences):,} parallel pairs from NLLB corpus...")
print("(Using real translations, not synthetic generation)")

bt_pairs = []
generation_log = []

for i, (src, tgt) in enumerate(zip(source_sentences, target_sentences_reference)):
    # Skip empty or very short sentences
    if len(src) < 5 or len(tgt) < 5:
        continue
    
    bt_pairs.append({
        "source": src,
        "target": tgt,
        "method": "nllb_parallel",  # Real parallel data, not back-translation
        "domain": "general"
    })
    
    generation_log.append({
        "index": i,
        "source": src,
        "target": tgt,
        "method": "nllb_parallel"
    })

print(f"\nPrepared {len(bt_pairs):,} parallel pairs from NLLB corpus")
print("This is REAL parallel data - higher quality than synthetic!")

## 5. Generate Domain-Specific Dialogues

Generate conversations specific to medical, legal, and humanitarian contexts.

In [ ]:
# ============================================================
# OPTIONAL: GENERATE DOMAIN-SPECIFIC DIALOGUES
# ============================================================
# Skip this if you want to train quickly with just the NLLB data
# Set GENERATE_DOMAIN_DIALOGUES = True if you want to add domain-specific content

GENERATE_DOMAIN_DIALOGUES = False  # Set to True to generate with Gemma (takes 1-2 hours)

dialogue_pairs = []

if GENERATE_DOMAIN_DIALOGUES:
    print("Generating domain-specific dialogues with Gemma...")
    print("This adds medical, legal, and humanitarian context sentences.")
    
    domain_prompts = {
        "medical": [
            "patient describing symptoms to a doctor",
            "doctor asking about medical history",
        ],
        "legal": [
            "explaining legal rights to a refugee",
            "asylum interview question and answer",
        ],
        "humanitarian": [
            "registration process at a camp",
            "asking about family members or dependents",
        ]
    }
    
    NUM_DIALOGUES_PER_DOMAIN = 100  # Small number for speed
    
    for domain, contexts in domain_prompts.items():
        print(f"\nGenerating {domain} dialogues...")
        for i in tqdm(range(NUM_DIALOGUES_PER_DOMAIN), desc=domain):
            context = contexts[i % len(contexts)]
            try:
                source_text = generate_domain_sentence(SOURCE_LANG, domain, context)
                if not source_text or len(source_text) < 5:
                    continue
                target_text = translate(source_text, SOURCE_LANG, TARGET_LANG, domain=domain)
                if not target_text or len(target_text) < 3:
                    continue
                dialogue_pairs.append({
                    "source": source_text,
                    "target": target_text,
                    "method": "domain_dialogue",
                    "domain": domain,
                })
            except Exception as e:
                continue
    
    print(f"\nGenerated {len(dialogue_pairs)} domain dialogue pairs")
else:
    print("Skipping domain dialogue generation (GENERATE_DOMAIN_DIALOGUES = False)")
    print("Using only NLLB parallel data for training.")
    print("\nTo add domain-specific content, set GENERATE_DOMAIN_DIALOGUES = True")
    print("This requires GPU and takes 1-2 hours.")

## 6. Save Generated Data

Save all generated pairs to files for training.

In [ ]:
# ============================================================
# SAVE GENERATED DATA
# ============================================================
# Combine all pairs
all_pairs = bt_pairs + dialogue_pairs
print(f"Total pairs: {len(all_pairs):,}")

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save parallel text files
source_file = OUTPUT_DIR / f"synthetic.{SOURCE_LANG}"
target_file = OUTPUT_DIR / f"synthetic.{TARGET_LANG}"
meta_file = OUTPUT_DIR / "synthetic.meta.jsonl"

with open(source_file, 'w', encoding='utf-8') as sf, \
     open(target_file, 'w', encoding='utf-8') as tf, \
     open(meta_file, 'w', encoding='utf-8') as mf:
    
    for pair in all_pairs:
        sf.write(pair["source"] + "\n")
        tf.write(pair["target"] + "\n")
        meta = {
            "method": pair.get("method", "unknown"),
            "domain": pair.get("domain", "general"),
            "context": pair.get("context", "")
        }
        mf.write(json.dumps(meta, ensure_ascii=False) + "\n")

print(f"\nSaved to {OUTPUT_DIR}:")
print(f"  - {source_file.name}: {len(all_pairs):,} lines")
print(f"  - {target_file.name}: {len(all_pairs):,} lines")
print(f"  - {meta_file.name}: metadata")

# Save generation log
log_file = OUTPUT_DIR / "generation_log.jsonl"
with open(log_file, 'w', encoding='utf-8') as f:
    for entry in generation_log:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")
print(f"  - {log_file.name}: generation log")

# Save summary
summary = {
    "generated_at": datetime.now().isoformat(),
    "source_lang": SOURCE_LANG,
    "target_lang": TARGET_LANG,
    "teacher_model": TEACHER_MODEL,
    "total_pairs": len(all_pairs),
    "back_translation_pairs": len(bt_pairs),
    "dialogue_pairs": len(dialogue_pairs),
    "domains": {
        domain: sum(1 for p in all_pairs if p.get("domain") == domain)
        for domain in ["humanitarian", "medical", "legal", "general"]
    }
}

summary_file = OUTPUT_DIR / "generation_summary.json"
with open(summary_file, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)
print(f"  - {summary_file.name}: summary")

In [ ]:
# ============================================================
# SUMMARY
# ============================================================
print("=" * 60)
print("SYNTHETIC CORPUS GENERATION COMPLETE")
print("=" * 60)
print(f"\nLanguage pair: {SOURCE_NAME} ({SOURCE_LANG}) -> {TARGET_NAME} ({TARGET_LANG})")
print(f"Teacher model: {TEACHER_MODEL}")
print(f"\nTotal pairs generated: {len(all_pairs):,}")
print(f"  - Back-translation: {len(bt_pairs):,}")
print(f"  - Domain dialogues: {len(dialogue_pairs):,}")
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"\n" + "=" * 60)
print("NEXT STEPS")
print("=" * 60)
print("1. (Optional) Run 03_corpus_validation.ipynb to filter low-quality pairs")
print("2. Run 04_unsloth_finetuning.ipynb to train the model")
print("3. Run 05_evaluation.ipynb to evaluate translation quality")
print("=" * 60)

# Clean up GPU memory
del model
del tokenizer
import gc
gc.collect()
torch.cuda.empty_cache()
print("\nGPU memory released.")